In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from py4stat import datania

# Load data
survey_data = datania.generate_household_survey(n_households=1200, seed=42)
df = pd.read_csv(survey_data)

print("="*70)
print("DNSO STATISTICAL BRIEF: Household Income by Province")
print("="*70)
print(f"Data source: Datania Household Health Survey")
print(f"Sample size: {len(df):,} households")
print(f"Reference period: 2025")
print("="*70)

# Province reference for full names
province_names = {
    'CTR': 'Central Province',
    'EST': 'Eastern Province',
    'LKS': 'Lakeside Province',
    'NTH': 'Northern Province',
    'STH': 'Southern Province',
    'WST': 'Western Province'
}

# Store results for the summary table
results = []

print("\n--- Detailed Provincial Analysis ---\n")

for province_code in sorted(df["province"].unique()):
    province_name = province_names.get(province_code, province_code)
    prov_df = df[df["province"] == province_code]

    income = prov_df["monthly_income"].to_numpy()
    weights = prov_df["survey_weight"].to_numpy()
    n = len(income)

    # Task 1: Weighted mean
    # YOUR CODE HERE
    weighted_mean = ___

    # Task 2: Confidence interval (using simplified SE)
    se = stats.sem(income)
    ci = stats.t.interval(0.95, n-1, weighted_mean, se)

    # Task 3: RSE and quality flag
    # YOUR CODE HERE
    rse = ___

    if rse > 25:
        quality = "Suppress"
        symbol = "⛔"
    elif rse > 15:
        quality = "Caution"
        symbol = "⚠️"
    elif rse > 5:
        quality = "Acceptable"
        symbol = "✓"
    else:
        quality = "High Quality"
        symbol = "★"

    results.append({
        'code': province_code,
        'name': province_name,
        'n': n,
        'mean': weighted_mean,
        'ci_lower': ci[0],
        'ci_upper': ci[1],
        'rse': rse,
        'quality': quality,
        'symbol': symbol
    })

    print(f"{symbol} {province_name} ({province_code})")
    print(f"   Sample size: {n}")
    print(f"   Weighted mean income: {weighted_mean:,.0f} DKW")
    print(f"   95% CI: ({ci[0]:,.0f} - {ci[1]:,.0f}) DKW")
    print(f"   RSE: {rse:.1f}% ({quality})")
    print()

# Task 4: Publication-ready summary table
print("\n" + "="*70)
print("SUMMARY TABLE: Mean Monthly Household Income by Province")
print("="*70)
print(f"{'Province':<20} {'Mean (DKW)':>12} {'95% CI':>20} {'RSE':>8} {'Quality':>12}")
print("-"*70)

for r in sorted(results, key=lambda x: x['mean'], reverse=True):
    ci_str = f"({r['ci_lower']:,.0f}-{r['ci_upper']:,.0f})"
    print(f"{r['name']:<20} {r['mean']:>12,.0f} {ci_str:>20} {r['rse']:>7.1f}% {r['symbol']} {r['quality']}")

print("-"*70)

# National weighted estimate
national_income = df["monthly_income"].to_numpy()
national_weights = df["survey_weight"].to_numpy()
national_mean = np.average(national_income, weights=national_weights)
national_se = stats.sem(national_income)
national_ci = stats.t.interval(0.95, len(national_income)-1, national_mean, national_se)
national_rse = (national_se / national_mean) * 100

print(f"{'NATIONAL'::<20} {national_mean:>12,.0f} ({national_ci[0]:,.0f}-{national_ci[1]:,.0f}):>20 {national_rse:>7.1f}% ★ High Quality")
print("="*70)

# Task 5: Brief interpretation
print("\n--- KEY FINDINGS ---")
print("""# YOUR INTERPRETATION HERE
# Write 2-3 sentences summarizing:
# - Which province has the highest/lowest income?
# - Are there any quality concerns?
# - What's the range of provincial estimates?
""")

# Suggested structure for interpretation:
highest = max(results, key=lambda x: x['mean'])
lowest = min(results, key=lambda x: x['mean'])
caution_provinces = [r for r in results if r['quality'] in ['Caution', 'Suppress']]

print(f"\n1. {highest['name']} has the highest mean income ({highest['mean']:,.0f} DKW),")
print(f"   while {lowest['name']} has the lowest ({lowest['mean']:,.0f} DKW).")
print(f"\n2. The income gap between highest and lowest provinces is {highest['mean']-lowest['mean']:,.0f} DKW.")

if caution_provinces:
    print(f"\n3. Quality note: {len(caution_provinces)} province(s) have elevated RSE values")
    print(f"   and should be interpreted with caution.")
else:
    print(f"\n3. All provincial estimates meet quality standards for publication.")